# LaGCO-RL setup and pretrained artifact installer

This notebook finalises the setup of dependencies, install notebook APIs, and downloads available pretrained artifacts for both MaxCut and VMP.



In [ ]:
from pathlib import Path
import subprocess, json, zipfile, shutil, urllib.request
import yaml

BASE_DIR = Path.cwd().resolve()
REPO_DIR = BASE_DIR / "LaGCORL"
REPO_URL = "https://github.com/terranovafr/LaGCO-RL.git"
print("Working directory:", BASE_DIR)
print("Repository target:", REPO_DIR)


## 1. Clone LaGCO-RL

We first clone the repository for the [LaGCO-RL library](https://github.com/terranovafr/LaGCO-RL).   


In [ ]:
if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise FileExistsError(f"{REPO_DIR} exists and is not a Git repository; move it or choose another BASE_DIR.")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

assert (REPO_DIR / "setup.sh").is_file()
print("Repository ready:", REPO_DIR)


## 2. Run `setup.sh` for LaGCO-RL

This runs the library setup script. The script may prompt for a Hugging Face API key; leave it blank if you do not need authenticated access.


In [ ]:
setup_script = REPO_DIR / "setup.sh"

process = subprocess.Popen(
    ["bash", str(setup_script)],
    cwd=REPO_DIR,
    stdin=subprocess.PIPE,
    text=True,
)

process.communicate(input="\n" * 100)

if process.returncode != 0:
    raise subprocess.CalledProcessError(process.returncode, process.args)

## 3. Update `config.yaml` without losing the setup key

`setup.sh` currently initializes `tsp.default_envs` and `huggingface.key`. This cell preserves those entries and adds `maxcut.default_envs: null` if needed. The value remains `null` until you create/download a compatible graph corpus; this notebook's direct test environment does not need a corpus for basic inference, but GAE training does.


In [ ]:
config_path = REPO_DIR / "config.yaml"
config = yaml.safe_load(config_path.read_text(encoding="utf-8")) or {}
config.setdefault("tsp", {}).setdefault("default_envs", None)
config.setdefault("maxcut", {}).setdefault("default_envs", None)
config.setdefault("vmp", {}).setdefault("default_envs", None)
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print("Updated config:", config_path)

## 4. Install the notebook helper modules

Keep `notebook_train_api.py`, `notebook_test_api.py`, `notebook_train_gae_api.py`, and `notebook_helpers_api.py` beside this setup notebook. The installer places project APIs under `LaGCORL/agents/`; the general helper module is also installed there for notebooks launched from the repository root.


In [ ]:
helper_sources = {
    "notebook_train_api.py": BASE_DIR / "notebook_train_api.py",
    "notebook_test_api.py": BASE_DIR / "notebook_test_api.py",
    "notebook_train_gae_api.py": BASE_DIR / "notebook_train_gae_api.py",
    "notebook_helpers_api.py": BASE_DIR / "notebook_helpers_api.py",
}
for filename, source_path in helper_sources.items():
    if not source_path.is_file():
        raise FileNotFoundError(
            f"Missing {source_path}. Place the supplied helper file beside this setup notebook."
        )
    destination = REPO_DIR / "agents" / filename
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_path, destination)
    print("Installed:", destination)


## 5. Configure and install pretrained artifacts for both benchmarks

Each ZIP is downloaded temporarily, then its complete payload is extracted directly into the corresponding `*_pretrained_*` run directory. If the ZIP contains one enclosing top-level folder, that folder is flattened. No `checkpoints/`, `tensorboard/`, `test_metrics/`, or `train_metrics/` folders are created, and the ZIP is not retained in the run directory.


In [ ]:
from datetime import datetime
import tempfile

LOG_ROOT = REPO_DIR.parent / "logs"
PRETRAINED_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

# Keep the same artifact types for each benchmark. Fill in a URL only when a
# checkpoint trained for that specific benchmark is actually available.
BENCHMARK_MODEL_URLS = {
    "maxcut": {
        "discrete": "https://www.dropbox.com/scl/fi/cz43fo89pjr9knm5e09bp/discrete_maxcut_pretrained.zip?rlkey=6c9oghniyntm0rdng6o32c5j7&st=zt1u1pvz&dl=1",       # SB3 PPO model.zip for ordinary discrete MaxCut.
        "gnn_discrete": "https://www.dropbox.com/scl/fi/72oeme885ls2fbgxsfvzv/gnn_discrete_maxcut_pretrained.zip?rlkey=gwjdbnrspbiclcwa6xpc8xe1s&st=nbyachwi&dl=1",   # Padded GNN+discrete PPO model.zip.
        "iterative": "https://www.dropbox.com/scl/fi/05dt93ek8o88bed5v9d4s/iterative_maxcut_pretrained.zip?rlkey=u72ys42k5n90q275sqrywaf1w&st=wylook85&dl=1",
        "projection": "https://www.dropbox.com/scl/fi/tgugm5tw0o0334fkbcliv/projection_maxcut_pretrained.zip?rlkey=65aswefsootayegnv2bvlaqhg&st=6otxuewh&dl=1",
        "gae_bundle": "https://www.dropbox.com/scl/fi/wecc06pjqaq38awov84zt/gae_maxcut_pretrained.zip?rlkey=b1y1f4d6ya3m1n3018lcrjlcx&st=0dxcg8e4&dl=1",
    },
    "vmp": {
        "discrete": "https://www.dropbox.com/scl/fi/0dnp7vildoero3hjcy572/discrete_vmp_pretrained.zip?rlkey=m4gyy5nf7gq0qz7ra7jxf05mh&st=8tago84x&dl=1",       # VMP-specific PPO model.zip.
        "gnn_discrete": "https://www.dropbox.com/scl/fi/04a793z972gi8dke5as60/gnn_discrete_vmp_pretrained.zip?rlkey=7gilw54mtyfm3uevw9jw0plpy&st=tmdd1tks&dl=1",   # VMP-specific GNN+discrete PPO model.zip, if supported.
        "iterative": "https://www.dropbox.com/scl/fi/1mnwi6dtfquhf9m6nurcj/iterative_vmp_pretrained.zip?rlkey=3rtsvp3h4yik4edvey96mg0ik&st=w6kd07uu&dl=1",      # VMP-specific iDQN checkpoint.
        "projection": "https://www.dropbox.com/scl/fi/opw949u7zzhxp1udulnuy/projection_vmp_pretrained.zip?rlkey=28jocuufltwer23s964dxh9y1&st=zt1vgemf&dl=1",     # VMP-specific projection PPO checkpoint.
        "gae_bundle": "https://www.dropbox.com/scl/fi/vrmqnq9untw9dvzgk9nww/gae_vmp_pretrained.zip?rlkey=j39dwa6z3fy5nie3sbbcvw0pp&st=wlx4hpj8&dl=1",     # VMP-specific GAE encoder bundle.
    },
}

# Map the logical artifact names to the same log folders used by the test notebook.
ARTIFACT_FOLDERS = {
    "discrete": "discrete_ppo",
    "gnn_discrete": "discrete_ppo_gnn_padded",
    "iterative": "iterative_dqn",
    "projection": "projection_ppo",
    "gae_bundle": "gae",
}

def download_file(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(url, headers={"User-Agent": "LaGCO-RL-notebook-setup"})
    with urllib.request.urlopen(request, timeout=120) as response, destination.open("wb") as out:
        shutil.copyfileobj(response, out)
    if not destination.is_file() or destination.stat().st_size == 0:
        destination.unlink(missing_ok=True)
        raise IOError(f"Downloaded an empty file from {url}")
    if not zipfile.is_zipfile(destination):
        destination.unlink(missing_ok=True)
        raise zipfile.BadZipFile(f"Downloaded file is not a valid ZIP archive: {url}")
    print(f"Downloaded temporary archive ({destination.stat().st_size:,} bytes)")
    return destination


def extract_zip_contents(archive_path, destination_dir):
    """Extract all ZIP contents directly into destination_dir.

    If the ZIP contains one enclosing top-level directory, its contents are
    flattened into destination_dir. The ZIP itself is not kept in the run folder.
    """
    archive_path = Path(archive_path)
    destination_dir = Path(destination_dir)
    destination_dir.mkdir(parents=True, exist_ok=True)

    if not zipfile.is_zipfile(archive_path):
        raise zipfile.BadZipFile(f"Downloaded file is not a valid ZIP archive: {archive_path}")

    with tempfile.TemporaryDirectory(prefix="pretrained_extract_") as temp_name:
        temp_dir = Path(temp_name)
        with zipfile.ZipFile(archive_path, "r") as zf:
            members = zf.infolist()
            root = temp_dir.resolve()

            # Validate every member before extracting (prevents ZIP-slip).
            for member in members:
                target = (temp_dir / member.filename).resolve()
                if target != root and root not in target.parents:
                    raise ValueError(f"Unsafe path in ZIP archive: {member.filename}")

            zf.extractall(temp_dir)

        # Flatten a single wrapper directory, if the archive has one.
        entries = list(temp_dir.iterdir())
        if len(entries) == 1 and entries[0].is_dir():
            payload_root = entries[0]
        else:
            payload_root = temp_dir

        # Move every payload item directly into the pretrained run directory.
        for item in payload_root.iterdir():
            target = destination_dir / item.name
            if target.exists():
                if target.is_dir():
                    shutil.rmtree(target)
                else:
                    target.unlink()
            shutil.move(str(item), str(target))

    extracted_files = [p for p in destination_dir.rglob("*") if p.is_file()]
    json_files = [p for p in extracted_files if p.suffix.lower() == ".json"]
    print(
        f"Extracted archive contents directly into {destination_dir} "
        f"({len(extracted_files)} files, {len(json_files)} JSON files)."
    )
    if json_files:
        print("  JSON metadata:")
        for path in json_files:
            print("   -", path.relative_to(destination_dir))

    return destination_dir


print("Logs root:", LOG_ROOT.resolve())
print("Configured artifact slots:")
for benchmark, artifacts in BENCHMARK_MODEL_URLS.items():
    print(f"  {benchmark}:")
    for artifact, url in artifacts.items():
        print(f"    {artifact}: {'URL configured' if url.strip() else 'skipped (no URL)'}")


In [ ]:
downloaded = {}
skipped = []
failed = []

# Each artifact gets one pretrained run directory containing the ZIP payload
# directly. Archives are downloaded to a temporary location and removed after
# successful extraction; no checkpoints/tensorboard/metrics folders are created.
for benchmark, artifact_urls in BENCHMARK_MODEL_URLS.items():
    run_name = f"{benchmark}_pretrained_{PRETRAINED_STAMP}"
    for artifact_name, url in artifact_urls.items():
        url = (url or "").strip()
        if not url:
            skipped.append((benchmark, artifact_name))
            print(f"Skipping {benchmark}/{artifact_name}: no URL configured.")
            continue

        agent_folder = ARTIFACT_FOLDERS[artifact_name]
        run_dir = LOG_ROOT / agent_folder / run_name
        run_dir.mkdir(parents=True, exist_ok=True)

        try:
            archive_name = "gae_bundle.zip" if artifact_name == "gae_bundle" else "model.zip"
            with tempfile.TemporaryDirectory(prefix="pretrained_download_") as temp_name:
                archive_path = Path(temp_name) / archive_name
                download_file(url, archive_path)
                installed_path = extract_zip_contents(archive_path, run_dir)

            key = f"{benchmark}_{artifact_name}"
            downloaded[key] = installed_path
            print(f"Installed {key}: {installed_path.resolve()}")
        except Exception as exc:
            failed.append((benchmark, artifact_name, str(exc)))
            print(f"FAILED {benchmark}/{artifact_name}: {type(exc).__name__}: {exc}")

print("\nDownload summary")
print("  Installed:", len(downloaded))
print("  Skipped (blank URL):", len(skipped))
print("  Failed:", len(failed))
if downloaded:
    for name, path in downloaded.items():
        print(f"  {name}: {path}")
if failed:
    print("\nSome configured downloads failed. Review the errors above and rerun this cell after fixing the URLs.")
